# Per-gene colocboost across contexts in a single study

## Description

For a single study's `QtlDataset` RDS, run `pecotmr::colocboostPipeline()` per focal gene. Each task colocalizes the focal gene's signal across the QtlDataset's contexts (the within-study, cross-context case — no GWAS input). Gene-level parallelization is via SoS task fan-out: every task loads the same RDS and runs one focal gene.

## Inputs

- `--qtl-dataset` &mdash; path to the QtlDataset RDS produced by `qtl_dataset.ipynb`.
- `--genes` &mdash; explicit gene-ID list (one focal gene per task).
- `--cis-window` &mdash; bp window around each focal gene's TSS for variant selection. Default 1,000,000.

## Output

- `{cwd}/{study}.{gene}.colocboost.rds` per gene.


## Example

```bash
sos run pipeline/colocboost.ipynb colocboost \
    --cwd output \
    --study ROSMAP_DLPFC \
    --qtl-dataset output/ROSMAP_DLPFC.qtl_dataset.rds \
    --genes ENSG00000060237 ENSG00000234593 \
    --cis-window 1000000
```


In [ ]:
[global]
parameter: cwd = path('output')
parameter: study = str
parameter: qtl_dataset = path
parameter: genes = []
parameter: cis_window = 1000000
parameter: container = ''
parameter: job_size = 1
parameter: walltime = '1h'
parameter: mem = '16G'
parameter: numThreads = 1


In [ ]:
[colocboost]
input: qtl_dataset, for_each = 'genes'
output: f"{cwd}/{study}.{_genes}.colocboost.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${path("code/script/pecotmr_integration/colocboost.R", absolute=True)} \
        --qtl-dataset ${_input} \
        --gene-id ${_genes} \
        --cis-window ${cis_window} \
        --output ${_output}
